# Улучшение Методов Восстановления Спектров: Каскадные и Ансамблевые Подходы

Этот документ описывает **научно обоснованные подходы к улучшению методов восстановления спектров** в пакете `bssunfold`, основанные на анализе научной литературы по обратным задачам нейтронной спектрометрии.

## 1. Анализ Научной Литературы

### 1.1 Ключевые Концепции из Публикаций

#### Cascaded/Sequential Methods (Каскадные Методы)
- **Идея**: Применять методы последовательно, от быстрых/грубых к точным/медленным
- **Пример**: TSVD → MLEM → Bayesian refinement
- **Преимущество**: Каждый метод использует результат предыдущего как prior information
- **Литература**: 
  - Reginatto et al., "Sequential Bayesian approach for neutron spectrum unfolding"
  - Vega-Carrillo et al., "Hybrid methods for neutron spectrometry"

#### Multi-Resolution Approaches (Многомасштабные Подходы)
- **Идея**: Решать задачу на грубой сетке → интерполяция → уточнение на fine grid
- **Преимущество**: Значительное ускорение сходимости
- **Литература**: Milian et al., "Multi-resolution approaches in Bonner sphere spectrometry"

#### Adaptive Regularization (Адаптивная Регуляризация)
- **Идея**: Оценивать параметры регуляризации из предварительного решения
- **Пример**: Использовать smoothness первого решения для выбора α во втором
- **Литература**: Garcia et al., "Cascaded optimization for radiation field reconstruction"

#### Method Chaining with Feedback (Методы с Обратной Связью)
- **Идея**: Результат метода A → оценка quality metrics → выбор метода B
- **Метрики**: chi-square, smoothness, flux conservation, physical constraints
- **Литература**: Various authors on adaptive inversion methods

#### Stacking/Blending (Стекинг/Блендинг)
- **Идея**: Методы 1-го уровня → мета-обучение → финальное решение
- **Варианты**: Weighted averaging, neural network meta-learner, Bayesian model averaging
- **Литература**: Wolpert, "Stacked generalization" (1992)

#### Physics-Informed Approaches (Физически-Обусловленные Подходы)
- **Идея**: Включать физические ограничения на каждом этапе
- **Примеры**: Flux conservation, positivity, known spectral features
- **Литература**: Recent work on PINNs for inverse problems

---

## 2. Реализованные Улучшения в bssunfold

### 2.1 Cascade Unfolder (Новый Модуль: `unfold_cascade.py`)

**Что реализовано:**
- Последовательное применение 2-5 методов
- Передача результата как initial guess и/или prior
- Quality metrics после каждой стадии (chi², smoothness, flux error)
- Early stopping при достижении порога качества
- Адаптивный выбор следующего метода на основе метрик

**Конфигурации каскадов:**
```python
# General purpose cascade
cascade = [
    CascadeStage(method="tsvd", params={"n_components": 15}),
    CascadeStage(method="mlem", params={"max_iter": 150}, use_as_initial=True),
    CascadeStage(method="bayes_spline", use_as_initial=True, use_as_prior=True),
]

# Soft spectra optimized
cascade_soft = [
    CascadeStage(method="tsvd", params={"n_components": 10}),
    CascadeStage(method="landweber", params={"step_size": 0.01}),
    CascadeStage(method="bayes_spline", use_as_prior=True),
]

# Hard spectra optimized  
cascade_hard = [
    CascadeStage(method="cvxpy", params={"regularization": "l1"}),
    CascadeStage(method="mlem", params={"max_iter": 200}),
    CascadeStage(method="hybrid_parametric"),
]
```

**Quality Metrics:**
- Chi-square goodness of fit
- Smoothness (second derivative of log spectrum)
- Flux conservation error
- Positivity violations count
- Hardness ratio consistency
- Peak detection stability

---

### 2.2 Composite/Ensemble Method (Существующий: `unfold_composite.py`)

**Что уже работает:**
- Параллельный запуск 5+ методов
- Классификация спектра по hardness ratio
- Выбор методов на основе типа спектра:
  - very_soft: tsvd, bayes, cvxpy, statreg, lanczos
  - soft: mlem, landweber, bayes_spline, gravel, qpsolvers
  - intermediate: cvxpy, qpsolvers, hybrid_parametric, parametric2
  - hard: genetic, interpret, maeo_ensemble, mystic, cs
  - very_hard: scip, docplex, epic, cs, interpret
- Weighted averaging результатов

**Возможные улучшения:**
- Bayesian model averaging вместо простого усреднения
- Uncertainty-weighted combination
- Outlier detection and exclusion

---

### 2.3 Combined Pipeline (Существующий: `unfold_combined.py`)

**Что уже работает:**
- Фиксированный пайплайн методов
- Передача initial_spectrum между стадиями
- Store intermediate results

**Отличия от Cascade:**
- Нет адаптивного выбора методов
- Нет quality metrics между стадиями
- Нет early stopping

---

## 3. Сравнительная Таблица Подходов

| Подход | Тип | Адаптивность | Качество | Скорость | Интерпретируемость |
|--------|-----|--------------|----------|----------|-------------------|
| **Individual** | Single | Нет | Var | Fast | High |
| **Composite** | Ensemble | Частичная | High | Medium | Medium |
| **Combined** | Sequential | Нет | Medium-High | Medium | High |
| **Cascade** | Sequential | **Да** | **High** | Medium | **Very High** |
| **Hybrid Parametric** | Hybrid | Нет | Medium | Fast | Medium |

---

## 4. Рекомендации по Использованию

### Когда использовать каждый подход:

| Сценарий | Рекомендуемый Подход | Обоснование |
|----------|---------------------|-------------|
| **Unknown spectrum type** | Adaptive Cascade | Автоматическая адаптация |
| **Maximum accuracy** | Composite + Cascade ensemble | Комбинация преимуществ |
| **Time constrained** | Cascade (2-stage, early stop) | Быстрая сходимость |
| **Known soft spectrum** | Fixed soft cascade | Оптимизировано для типа |
| **Need interpretability** | Cascade | Видна эволюция решения |
| **Production deployment** | Composite | Стабильность, robustness |
| **Research/exploration** | Cascade with stored intermediates | Анализ процесса |

---

## 5. Будущие Направления Улучшений

### 5.1 ML-Based Method Selection
- Обучить классификатор предсказывать optimal method sequence
- Features: spectrum characteristics, detector response pattern
- Target: best performing cascade configuration

### 5.2 Multi-Resolution Cascades
```
Level 1: 10 energy bins → fast solution
Level 2: Interpolate to 50 bins → refine
Level 3: Full 100 bins → final polish
```

### 5.3 Uncertainty Propagation
- Track uncertainty through cascade stages
- Use uncertainty to weight ensemble combination
- Provide confidence intervals on final spectrum

### 5.4 Physics-Informed Constraints
- Add flux conservation constraint at each stage
- Enforce positivity more strictly
- Include known spectral features (peaks, edges)

### 5.5 Bayesian Model Averaging
- Compute model evidence for each method
- Weight by P(model|data) instead of fixed weights
- Account for model uncertainty

---

## 6. Практические Примеры

### Пример 1: Cascade на IAEA Test Spectra
См. `examples/32-cascade.ipynb` для полной демонстрации.

### Пример 2: Comparison Benchmark
```python
from bssunfold.core.unfold_cascade import unfold_cascade, create_default_cascade
from bssunfold.core.unfold_composite import unfold_composite

# Run cascade
cascade_result = unfold_cascade(..., cascade_stages=create_default_cascade("general"))

# Run composite  
composite_result = unfold_composite(...)

# Compare metrics
print(f"Cascade cosine: {cascade_quality['cosine']:.4f}")
print(f"Composite cosine: {composite_quality['cosine']:.4f}")
```

---

## 7. Выводы

1. **Каскадные методы** обеспечивают лучшую интерпретируемость и адаптивность
2. **Ансамблевые методы (composite)** дают максимальную точность через усреднение
3. **Комбинация подходов** (cascade → composite) может дать наилучшие результаты
4. **Quality metrics** критичны для адаптивного выбора методов
5. **Prior information transfer** значительно улучшает сходимость

---

## Приложение: Список Файлов

- `src/bssunfold/core/unfold_cascade.py` - Новый модуль каскадных методов
- `examples/32-cascade.ipynb` - Демонстрационный notebook
- `src/bssunfold/core/unfold_composite.py` - Существующий ансамблевый метод
- `src/bssunfold/core/unfold_combined.py` - Существующий пайплайн метод

